# Initialization

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
import nglview as nv
import numpy as np
import os
import mdtraj as md
import io

In [ ]:
# cd into a new directory
!mkdir output
%cd output

In [ ]:
# copy necessary files and visualize
!mkdir data
%cd data
!cp ../../data/ethanol.gro ../../data/topol.top .
!ls
%cd ..

In [ ]:
# Visualize ethanol structure

# Coordinate system conversion
def Visualize(groPath, topPath):
    !mkdir viz
    !cp $groPath viz/mol.gro
    !cp $topPath viz/topol.top
    %cd viz
    !touch viz.mdp
    !gmx grompp -f viz.mdp -c mol.gro -p topol.top -o topol.tpr >> /dev/null 2>&1
    !printf "0\n" | gmx trjconv -f mol.gro -s topol.tpr -o viz.gro -pbc mol -ur compact >> /dev/null 2>&1

    view = nv.show_structure_file("viz.gro")
    %cd ..
    view.camera = "orthographic"
    return view

%cd data
view = nv.show_structure_file("ethanol.gro")
view.camera = "orthographic"
%cd ..

view

In [ ]:
# Define a simulation box and fill it with water
!mkdir solvated
%cd solvated
!cp ../data/ethanol.gro ../data/topol.top .

!gmx editconf -f ethanol.gro -o box.gro -bt dodecahedron -d 1.2
!gmx solvate -cp box.gro -cs -o solvated.gro -p topol.top
!ls

%cd ..

In [ ]:
!ls --tree

In [ ]:
view = Visualize("./solvated/solvated.gro", "./solvated/topol.top")
view.add_representation(repr_type='ball+stick', selection='SOL',opacity=0.5)
view

In [ ]:
# Energy minimization

def write_mdp(mdp_string, mdp_name):
    mdp_filehandle=open(mdp_name,'w')
    mdp_filehandle.write(mdp_string)
    mdp_filehandle.close()

!mkdir em
%cd em

em_mdp="""; energy minimization
integrator  = steep
nsteps      = 5000
coulombtype = pme
"""
write_mdp(em_mdp, "em.mdp")

!cp ../solvated/solvated.gro ../solvated/topol.top .
!gmx grompp -f em.mdp -c solvated.gro -o em.tpr
!gmx mdrun -reprod -deffnm em -ntmpi 1 -ntomp 8

%cd ..

In [ ]:
view = Visualize("./em/em.gro", "./em/topol.top")
view.add_representation(repr_type='ball+stick', selection='SOL',opacity=0.5)
view

In [ ]:
def ReadXVG(path, names):
    content = None
    with open(path, "r") as file:
        content = file.readlines()
        content = [c for c in content if not c.startswith("#") and not c.startswith("@")]
        content = "".join(content)
    df = pd.read_csv(io.StringIO(content), sep='\\s+', header=None, names=names)
    return df

%cd em
!printf "Potential\n0\n" | gmx energy -f em.edr -o potential.xvg -xvg none
df = ReadXVG("potential.xvg", ["Step", "Potential energy"])
%cd ..

steps, potential = df.loc[:,"Step"].to_numpy(), df.loc[:,"Potential energy"].to_numpy()

ignoreSteps = 100
steps = steps[ignoreSteps:]
potential = potential[ignoreSteps:]
    
fig, ax = plt.subplots(figsize=(8,6))
ax.set_title("Potential energy during EM")
ax.set_ylabel("Potential energy (kJ/mol)")
ax.set_xlabel("Step")
ax.plot(steps, potential, label="Potential energy")
ax.legend()
plt.show()
plt.close()

In [ ]:
# Equilibration

!mkdir equil
%cd equil
!cp ../em/em.gro ../em/topol.top .

equil_mdp=""";equilibration mdp options
integrator               = md
nsteps                   = 100000
dt                       = 0.002
nstenergy                = 100
nstxout-compressed       = 50
rlist                    = 1.1
nstlist                  = 10
rvdw                     = 1.1
coulombtype              = pme
rcoulomb                 = 1.1
fourierspacing           = 0.13
constraints              = h-bonds
tcoupl                   = v-rescale
tc-grps                  = system
tau-t                    = 0.5
ref-t                    = 300
pcoupl                   = C-rescale
ref-p                    = 1
compressibility          = 4.5e-5
tau-p                    = 1
gen-vel                  = yes
gen-temp                 = 300
"""
write_mdp(equil_mdp, "equil.mdp")
!gmx grompp -f equil.mdp -c em.gro -o equil.tpr
!gmx mdrun -reprod -deffnm equil -ntmpi 1 -ntomp 8 -v

%cd ..

# Free energy calculation

In [ ]:
# Create mdp files
run_mdp="""; Free energy calculation
integrator               = sd
nsteps                   = 100000
dt                       = 0.002
nstenergy                = 1000
nstcalcenergy            = 50 ; should be a divisor of nstdhdl 
nstlog                   = 5000

; cut-offs at 1.0nm
rlist                    = 1.1
rvdw                     = 1.1
; Coulomb interactions
coulombtype              = pme
rcoulomb                 = 1.1
fourierspacing           = 0.13
; Constraints
constraints              = h-bonds

; T = 300 K
tc-grps                  = system
tau-t                    = 2.0
ref-t                    = 300
; p = 1 bar
pcoupl                   = C-rescale
ref-p                    = 1.0
compressibility          = 4.5e-5
tau-p                    = 5.0

; free energy params
free-energy              = yes
couple-moltype           = ethanol
nstdhdl                  = 50

sc-power                 = 1
sc-sigma                 = 0.3
sc-alpha                 = 1.0

couple-intramol          = no
couple-lambda0           = none
couple-lambda1           = vdwq

coul-lambdas             = {l_q}
vdw-lambdas              = {l_vdw}
fep-lambdas              = {l_fep}
init-lambda-state        = {state}
"""

mdp_files = []
lambdasVDW = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
lambdasQ   = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
lambdasFEP = [0.0, 0.1, 0.2, 0.2, 0.3, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

nl = len(lambdasVDW)
lqs = " ".join(str(l) for l in lambdasQ)
lvdws = " ".join(str(l) for l in lambdasVDW)
lfeps = " ".join(str(l) for l in lambdasFEP)

for i in range(nl):
    mdp_files.append(run_mdp.format(l_q=lqs, l_vdw=lvdws, l_fep=lfeps, state=nl-1-i))

In [ ]:
# Create directories

dirFormat = "lambda_{0:0>4}"

for i in range(len(mdp_files)):
    dirname = "lambdas/"+dirFormat.format(i)
    os.makedirs(dirname, exist_ok=True)
    write_mdp(mdp_files[i], os.path.join(dirname, "grompp.mdp"))
    !cp equil/topol.top $dirname/topol.top

!cat lambdas/lambda_0000/grompp.mdp

In [ ]:
# Run sequential simulations

!cp equil/equil.gro lambdas/lambda_0000/conf.gro

%cd lambdas

for i in range(len(mdp_files)):
    dirname = dirFormat.format(i)
    dirnameNext = dirFormat.format(i+1)
    
    %cd $dirname
    !gmx grompp
    !gmx mdrun -reprod -ntmpi 1 -ntomp 8 -v
    %cd ..

    if i+1 < len(mdp_files):
        !cp $dirname/confout.gro $dirnameNext/conf.gro

%cd ..

In [ ]:
# Free energy computation

!mkdir fe
%cd fe
    
bar_string = " ".join(
    os.path.join("../lambdas", dirFormat.format(i), "dhdl.xvg") 
    for i in range(nl))
!gmx bar -b 50 -f $bar_string -prec 6 -o values.xvg

%cd ..

In [ ]:
# Convert to kJ/mol

%cd fe
df = ReadXVG("values.xvg", ["step", "DG", "err"])
%cd ..

k = 1.380649e-23
Na = 6.02214076e23
Temp = 300
rescale = k * Temp * Na / 1000

df.loc[:,["DG","err"]] *= rescale

print(df)

fig, ax = plt.subplots(figsize=(12,4))

bar = ax.bar(df.loc[:,"step"], df.loc[:,"err"], label="Error")

tx = ax.twinx()

pvdw, = tx.plot(np.arange(nl), lambdasVDW, "k.-", label="VDW lambda")
pq, = tx.plot(np.arange(nl), lambdasQ, "r.-", label="Coulomb lambda")

ax.legend(handles=[bar, pvdw, pq], loc="upper left")

ax.set_title("DG error and coupling lambdas")
ax.set_xlabel("Step")
ax.set_ylabel("Error")

tx.set_ylabel("Lambda")

fig.savefig("../figures/errors.png")

plt.show()
plt.close()